In [1]:
import pandas as pd
import logging
from pathlib import Path

In [10]:
file_path = Path("../data/raw/data1.csv")

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [4]:
def csv_reader(path: Path) -> pd.DataFrame:
    
    logger.info(f"Loading csv from: {path}")

    if not path.exists():
        logger.error(f"File not found: {path}")
        raise FileNotFoundError(f"Check the file path!")
    df = pd.read_csv(path)
    logger.info(f"Loaded csv.")
    return df

In [11]:
df = csv_reader(file_path)

2026-04-30 15:42:30,724 - INFO - Loading csv from: ../data/raw/data1.csv
2026-04-30 15:42:30,742 - INFO - Loaded csv.


In [12]:
df.head(3)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


In [13]:
df.shape

(1058, 35)

In [16]:
df.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

In [17]:
df.dtypes

Age                         int64
Attrition                   int64
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeCount               int64
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                        str
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                       str
JobSatisfaction             int64
MaritalStatus                 str
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
Over18                        str
OverTime                      str
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StandardHours               int64
StockOptionLevel            int64
TotalWorkingYears           int64
TrainingTimesL

In [23]:
logger.info(f"Removing columns (only one unique value): Over18, StandardHours, EmployeeCount")
df.drop(columns=["Over18", "StandardHours", "EmployeeCount"], inplace=True)

2026-04-30 16:08:23,800 - INFO - Removing columns (only one unique value): Over18, StandardHours, EmployeeCount


In [24]:
logger.info("Standardize all object columns.")
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()

2026-04-30 16:11:44,658 - INFO - Standardize all object columns.
/var/folders/hv/wjvkzpmj6fv44k0l243tmj_h0000gn/T/ipykernel_28945/1563062672.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [25]:
logger.info("Rename column names to snake case.")
df.columns = df.columns.str.lower()

2026-04-30 16:12:38,999 - INFO - Rename column names to snake case.


In [27]:
logger.info("Installing packages: psycopg2-binary sqlalchemy")
%pip install psycopg2-binary sqlalchemy

2026-04-30 16:28:39,831 - INFO - Installing packages: psycopg2-binary sqlalchemy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 11.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
logger.info("Installing packages: python-dotenv")
%pip install -q python-dotenv

2026-04-30 16:32:07,583 - INFO - Installing packages: python-dotenv



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

import os
from dotenv import load_dotenv

load_dotenv("../.env")
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

DB_URL = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=os.getenv("SUPABASE_PASSWORD"),
    host="db.vqssaigndessrttplwib.supabase.co",
    port=5432,
    database="postgres"
)
try:
    engine = create_engine(DB_URL)
    df.to_sql("hr_attrition", engine, if_exists="replace", index=False)
    logger.info("Data loaded to database successfully")
except Exception as e:
    logger.error(f"Failed to load data: {e}")
    raise

2026-04-30 16:39:17,634 - INFO - Data loaded to database successfully


In [31]:
SUPABASE_PASSWORD

'Etuka_2004@@'